# Data loading

In [1]:
# Load data
import pandas as pd

SHORT = False

DATA_DIR = '../data'

if SHORT:
    OUTPUT_FILE = 'electricity_clean_short'
    METADATA_OUTPUT_FILE = 'metadata_clean_short'
    df_elec = pd.read_csv(DATA_DIR + '/electricity_short.csv', parse_dates=['timestamp'], index_col='timestamp')
else:
    OUTPUT_FILE = 'electricity_clean'
    METADATA_OUTPUT_FILE = 'metadata_clean'
    df_elec = pd.read_csv(DATA_DIR + '/electricity.csv', parse_dates=['timestamp'], index_col='timestamp')

# Load the metadata file
df_metadata = pd.read_csv(DATA_DIR + '/metadata.csv')

print("Data loaded")

Data loaded


# Filtering out useless building types

Might remove later

In [2]:
# Remove unnecessary data (Unimportant building types)

# Building types
target_types = [
    'parking', 
    'lodging', 
    'office', 
    'education', 
    # 'retail', 
    'assembly', 
    # 'other',
    'public', 
    # 'warehouse', 
    'food', 
    'utility', 
    'health', 
    # 'religion', 
    'science',
    # 'industrial', 
    # 'services', 
    # 'unknown',
]

# Filter columns that contain any of your target types
keep_columns = [col for col in df_elec.columns 
                if any(t in col.lower() for t in target_types)]

# Overwrite or create a filtered dataframe
df_filtered = df_elec[keep_columns].copy()

print(f"Buildings kept: {len(df_filtered.columns)} out of {len(df_elec.columns)}")

Buildings kept: 1485 out of 1578


# Remove buildings with no lat/lng

In [3]:
import pandas as pd

start_df = df_filtered

# 2. Identify "Geo-Valid" buildings
# We look for rows where both 'lat' AND 'lng' are not null
mask_has_coords = df_metadata['lat'].notnull() & df_metadata['lng'].notnull()
valid_building_ids = df_metadata.loc[mask_has_coords, 'building_id'].unique()

# 3. Filter the electricity DataFrame
# We keep only columns that exist in our 'valid_building_ids' list
# Note: We keep 'timestamp' if it is a column, or ignore it if it's the index
current_cols = start_df.columns
cols_to_keep = [col for col in current_cols if col in valid_building_ids]

# Apply the filter
df_geo_filtered = start_df[cols_to_keep]

# --- Reporting ---
original_count = len(current_cols)
final_count = len(df_geo_filtered.columns)
dropped_count = original_count - final_count

print("--- GEOSPATIAL FILTERING REPORT ---")
print(f"Buildings in electricity data:      {original_count}")
print(f"Buildings with valid Lat/Long:      {final_count}")
print(f"Buildings dropped (No coordinates): {dropped_count}")

# Check for name mismatches
if dropped_count == original_count:
    print("\nWARNING: All buildings were dropped. Check if 'building_id' in "
          "metadata matches the column names in your electricity data.")

--- GEOSPATIAL FILTERING REPORT ---
Buildings in electricity data:      1485
Buildings with valid Lat/Long:      1285
Buildings dropped (No coordinates): 200


# Remove zeros

Replace zero values with NAN

In [4]:
# Remove zeroes since elec use cant be zero
import numpy as np

start_df = df_geo_filtered

zeros_remaining = (start_df == 0).sum().sum()
print(f"Zeroes at beginning: {zeros_remaining}")
negatives_remaining = (start_df < 0).sum().sum()
print(f"Negatives at beginning: {negatives_remaining}")
start_nans = start_df.isnull().sum().sum()

# Replace all exact 0 values with NaN
df_no_zero = start_df.replace(0, np.nan)

# Replace negatives (No negatives, but keeping in case)
df_no_zero[df_no_zero < 0] = np.nan

print()
# Zeros remaining (Should be 0)
zeros_remaining = (df_no_zero == 0).sum().sum()
print(f"Zeros remaining in dataset: {zeros_remaining}")

# Negatives remaining
negatives_found = (df_no_zero < 0).sum().sum()
print(f"Negatives remaining: {negatives_found}")

# Check how many NaNs I just created
end_nans = df_no_zero.isnull().sum().sum()
new_nans = end_nans - start_nans
print(f"New NaN values added to dataset: {new_nans}")
print(f"Total NaN values now in dataset: {end_nans}")

Zeroes at beginning: 942958
Negatives at beginning: 0

Zeros remaining in dataset: 0
Negatives remaining: 0
New NaN values added to dataset: 942958
Total NaN values now in dataset: 1764844


# Remove outliers

Remove outliers by hour because large fluctuations between noon and midnight would cause issues
Difference between summer and winter is small enough that I am ignoring for now

In [5]:
# Remove outliers by hour
def remove_hourly_outliers(df, threshold=3.5):
    # 1. Group by the hour of the day
    group = df.groupby(df.index.hour)
    
    # 2. Calculate Median and MAD for every hour-group
    median = group.transform('median')
    mad = (df - median).abs().groupby(df.index.hour).transform('median')
    
    # 3. Calculate Modified Z-Score
    # Avoid division by zero if MAD is 0 (stagnant data)
    mod_z = 0.6745 * (df - median) / (mad + 1e-6)
    
    # 4. Mask the outliers
    return df.mask(mod_z.abs() > threshold)

start_df = df_no_zero

# NAN stuff
start_nans = start_df.isnull().sum().sum()

df_no_outliers = remove_hourly_outliers(start_df, threshold=3.5)
# Check how many NaNs we just created
end_nans = df_no_outliers.isnull().sum().sum()
new_nans = end_nans - start_nans
print(f"New NaN values added to dataset: {new_nans}")
print(f"Total NaN values now in dataset: {end_nans}")

New NaN values added to dataset: 675787
Total NaN values now in dataset: 2440631


# Interpolate

Calculate values for small holes of NAN values linearly based on values surrounding

In [6]:
# Interpolation
# This fills the "holes" left by both the zeros and the spikes
GAP_SIZE = 4

start_nans = df_no_outliers.isnull().sum().sum()

# Interpolate
df_interpolated = df_no_outliers.interpolate(method='linear', limit=GAP_SIZE, limit_direction='both')

end_nans = df_interpolated.isnull().sum().sum()
removed_nans = start_nans - end_nans
print(f"NAN values interpolated: {removed_nans}")
print(f"Total NaNs remaining (gaps > {GAP_SIZE}hr): {end_nans}")

NAN values interpolated: 595969
Total NaNs remaining (gaps > 4hr): 1844662


# Remove bad buildings

Remove buildings with more than 10% of data missing
Also remove buildings with no lat/lon

In [7]:
# Set threshold percentage
THRESHOLD = 10

# Calculate the percentage of missing values per building
missing_pct = df_interpolated.isnull().mean() * 100

# Keep only buildings that have at least 90% of their data intact
keep_cols = missing_pct[missing_pct < THRESHOLD].index
df_removed_bad = df_interpolated[keep_cols]

print(f"Buildings dropped for excessive missingness: {len(missing_pct) - len(keep_cols)}")
print(f"Buildings remaining: {len(df_removed_bad.columns)}")

Buildings dropped for excessive missingness: 258
Buildings remaining: 1027


# Save output

In [8]:
# Save as a CSV
file_path_csv = DATA_DIR + '/' + OUTPUT_FILE + '.csv'

df_clean = df_removed_bad

# index=True is vital because your 'timestamp' is in the index
df_clean.to_csv(file_path_csv, index=True)

print(f"Data successfully saved to: {file_path_csv}")

# Save as parquet (Smaller and faster)
file_path_parquet = DATA_DIR + '/' + OUTPUT_FILE + '.parquet'

# engine='pyarrow' is fast and reliable
df_clean.to_parquet(file_path_parquet, engine='pyarrow')

print(f"Data successfully saved to: {file_path_parquet}")

Data successfully saved to: ../data/electricity_clean.csv
Data successfully saved to: ../data/electricity_clean.parquet


In [9]:
# Save metadata
import time
import pandas as pd

df_meta = df_metadata

# 1. Identify the "Survivor" buildings from your cleaned electricity data
# We ignore 'timestamp' if it's in the columns; otherwise, take all column names
survivor_buildings = df_clean.columns.tolist()

# 2. Filter the original metadata to only include these buildings
# This ensures a 1-to-1 match for your future Merges
df_metadata_clean = df_meta[df_meta['building_id'].isin(survivor_buildings)]

# --- Save the Cleaned Metadata ---
metadata_output_path = DATA_DIR + '/' + METADATA_OUTPUT_FILE + '.csv'
df_metadata_clean.to_csv(metadata_output_path, index=False)

# --- Final Sync Report ---
print("-" * 30)
print(f"Electricity Columns: {len(df_clean.columns)}")
print(f"Metadata Rows:      {len(df_metadata_clean)}")
print(f"Cleaned Metadata saved to: {metadata_output_path}")
print("-" * 30)

------------------------------
Electricity Columns: 1027
Metadata Rows:      1027
Cleaned Metadata saved to: ../data/metadata_clean.csv
------------------------------


# Final statistics

In [10]:
# display(df_clean.describe())
print("Unique primary usages")
print(df_metadata_clean['primaryspaceusage'].unique())

print("Unique sub primary usages")
print(df_metadata_clean['sub_primaryspaceusage'].unique())

Unique primary usages
['Education' 'Lodging/residential' 'Entertainment/public assembly'
 'Public services' 'Office' 'Food sales and service' 'Parking'
 'Healthcare' 'Utility' 'Technology/science']
Unique sub primary usages
['College Classroom' 'College Laboratory' 'Dormitory' 'Gymnasium'
 'Library' 'Office' 'Police Station' 'Museum' 'Auditorium'
 'Fitness Center' 'Restaurant' 'Student Union' 'Athletic Field'
 'Sports Stadium' 'Parking Garage' 'Health Services' 'Theater'
 'Swimming Pool' 'Planetarium' 'Primary/Secondary Classroom'
 'Central Plant' 'Social/Meeting Hall' 'Other - Recreation' 'K-12 School'
 'Fire Station' 'Other - Public Services' 'Other - Education'
 'Other - Lodging/Residential' 'Hospital (General Medical & Surgical)'
 'Other - Entertainment/Public Assembly' 'Parking'
 'Urgent Care Center/Clinic/Other Outpatient Office'
 'Prison/Incarceration' 'Senior Care Community' 'Other/Specialty Hospital'
 'Education' 'Technology/science' 'Lodging/residential' 'Public services'
 'E